In [22]:
import os

os.chdir(r"C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price")

print("Current folder:", os.getcwd())
print("Files in data folder:", os.listdir("data"))

Current folder: C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price
Files in data folder: ['data_kaggle.csv', 'Strict_Kaduna_South_Dataset.csv']


## Step 1: Load Datasets

We load both the Malaysia dataset and the Kaduna South dataset.

In [23]:
import pandas as pd

malaysia = pd.read_csv("data/data_kaggle.csv")
kaduna = pd.read_csv("data/Strict_Kaduna_South_Dataset.csv")

print("Malaysia shape:", malaysia.shape)
print("Kaduna shape:", kaduna.shape)

Malaysia shape: (53883, 8)
Kaduna shape: (37, 11)


## Step 2: Select Relevant Features

We select only the necessary columns required for modeling.

In [24]:
malaysia = malaysia[["Rooms", "Bathrooms", "Size", "Property Type", "Price"]]

kaduna = kaduna[["Bedrooms", "Bathrooms", "Size_sqm", "Property_Type", "Price"]]

## Step 3: Align Column Names

We rename Malaysia columns to match Kaduna dataset for consistency.

In [25]:
malaysia = malaysia.rename(columns={
    "Rooms": "Bedrooms",
    "Size": "Size_sqm",
    "Property Type": "Property_Type"
})

## Step 4: Clean Bedroom Values

Convert values such as "2+1" and "Studio" into numeric values.

In [26]:
import re
import numpy as np

def clean_bedrooms(value):
    value = str(value).lower().strip()

    if value == "studio":
        return 1

    numbers = re.findall(r"\d+", value)

    if len(numbers) == 0:
        return np.nan

    return sum([float(n) for n in numbers])

malaysia["Bedrooms"] = malaysia["Bedrooms"].apply(clean_bedrooms)
kaduna["Bedrooms"] = kaduna["Bedrooms"].apply(clean_bedrooms)

## Step 5: Clean Size Values

Extract numeric size values and convert square feet to square meters where necessary.

In [27]:
def clean_size(value):
    value = str(value).lower().strip()

    numbers = re.findall(r"\d+", value)

    if len(numbers) == 0:
        return np.nan

    size = float(numbers[0])

    if "sqft" in value or "sq ft" in value:
        size = size * 0.0929

    return size

malaysia["Size_sqm"] = malaysia["Size_sqm"].apply(clean_size)
kaduna["Size_sqm"] = kaduna["Size_sqm"].apply(clean_size)

## Step 6: Clean Price Values

Remove currency symbols and convert price into numeric values.

In [28]:
def clean_price(value):
    value = str(value).lower().strip()

    value = value.replace("rm", "")
    value = value.replace(",", "")
    value = value.replace("₦", "")

    numbers = re.findall(r"\d+", value)

    if len(numbers) == 0:
        return np.nan

    return float(numbers[0])

malaysia["Price"] = malaysia["Price"].apply(clean_price)
kaduna["Price"] = kaduna["Price"].apply(clean_price)

## Step 7: Convert Remaining Columns to Numeric

In [29]:
malaysia["Bathrooms"] = pd.to_numeric(malaysia["Bathrooms"], errors="coerce")
kaduna["Bathrooms"] = pd.to_numeric(kaduna["Bathrooms"], errors="coerce")

## Step 8: Remove Invalid Rows

Rows missing critical values are removed.

In [30]:
malaysia = malaysia.dropna(subset=["Bedrooms", "Bathrooms", "Size_sqm", "Price"])
kaduna = kaduna.dropna(subset=["Bedrooms", "Bathrooms", "Size_sqm", "Price"])

print("Malaysia cleaned:", malaysia.shape)
print("Kaduna cleaned:", kaduna.shape)

Malaysia cleaned: (50573, 5)
Kaduna cleaned: (37, 5)


## Step 9: Split Features and Target

In [31]:
X_malaysia = malaysia.drop(columns=["Price"])
y_malaysia = malaysia["Price"]

X_kaduna = kaduna.drop(columns=["Price"])
y_kaduna = kaduna["Price"]

## Step 10: Encode Categorical Variables

Convert Property_Type into numeric format using one-hot encoding.

In [32]:
combined = pd.concat([X_malaysia, X_kaduna])

combined = pd.get_dummies(combined, columns=["Property_Type"])

X_malaysia = combined.iloc[:len(X_malaysia)]
X_kaduna = combined.iloc[len(X_malaysia):]

print("Encoding complete")
print("Malaysia shape:", X_malaysia.shape)
print("Kaduna shape:", X_kaduna.shape)

Encoding complete
Malaysia shape: (50573, 102)
Kaduna shape: (37, 102)


Everthing aligned properly

In [33]:
print(X_malaysia.columns.equals(X_kaduna.columns))

True


VERIFY no missing values

In [34]:
print(X_malaysia["Bedrooms"].isna().sum())
print(X_kaduna["Bedrooms"].isna().sum())

0
0


## Converting Bedrooms to Integer

After cleaning, bedroom values are converted to integers for better clarity and consistency.

In [35]:
X_malaysia["Bedrooms"] = X_malaysia["Bedrooms"].astype(int)
X_kaduna["Bedrooms"] = X_kaduna["Bedrooms"].astype(int)

In [36]:
print(X_malaysia["Bedrooms"].head())
print(X_malaysia["Bedrooms"].dtype)

0    3
1    6
2    3
4    5
5    6
Name: Bedrooms, dtype: int64
int64


## Step 11: Train Model on Malaysia Data

In [37]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor()

model.fit(X_malaysia, y_malaysia)

print("Model training complete")

Model training complete


## Step 12: Split Kaduna Data for Calibration and Testing

In [38]:
from sklearn.model_selection import train_test_split

X_k_train, X_k_test, y_k_train, y_k_test = train_test_split(
    X_kaduna, y_kaduna, test_size=0.3, random_state=42
)

## Step 13: Calibration (Domain Adaptation)

Adjust predictions to match Kaduna price patterns.

In [39]:
from sklearn.linear_model import LinearRegression

pred_train = model.predict(X_k_train)

calibrator = LinearRegression()
calibrator.fit(pred_train.reshape(-1, 1), y_k_train)

print("Calibration complete")

Calibration complete


## Step 14: Evaluate Model Performance

In [40]:
from sklearn.metrics import mean_absolute_error, r2_score

pred_test = model.predict(X_k_test)

pred_calibrated = calibrator.predict(pred_test.reshape(-1, 1))

mae_before = mean_absolute_error(y_k_test, pred_test)
mae_after = mean_absolute_error(y_k_test, pred_calibrated)

r2_before = r2_score(y_k_test, pred_test)
r2_after = r2_score(y_k_test, pred_calibrated)

print("MAE Before:", mae_before)
print("MAE After:", mae_after)

print("R2 Before:", r2_before)
print("R2 After:", r2_after)

MAE Before: 62078292.06305418
MAE After: 28138183.72265302
R2 Before: -0.41411814033470495
R2 After: 0.5917980879297511


Save model

In [41]:
import joblib
joblib.dump(model, "model.pkl")

['model.pkl']

Save results

In [42]:
results = pd.DataFrame({
    "Actual": y_k_test,
    "Predicted": pred_calibrated
})

results.to_csv("outputs/results.csv", index=False)